# Heartbreak Hill Has Never Met Hurricane Point

The Big Sur Marathon course, its climb, and Heartbreak Hill at the same scale. Run `fetch_week1.py day2` first.

## 1 · Settings

In [ ]:
DAY      = 2
PROMPT   = "Lines"
SLUG     = "lines-big-sur"
NOTEBOOK = "day02_lines_big_sur.ipynb"
TITLE    = "Heartbreak Hill\nHas Never Met\nHurricane Point"
CREDITS  = "Course checkpoints: Big Sur International Marathon. Route: © OpenStreetMap contributors. Elevation: USGS 3DEP."

## 2 · Load

In [ ]:
import numpy as np, pandas as pd
from matplotlib.colors import LightSource
from scipy import ndimage
from mapkit import poster, publish
from mapkit.paths import PROCESSED

route = pd.read_csv(PROCESSED / "bigsur_profile.csv")
boston = pd.read_csv(PROCESSED / "boston_comm_ave.csv")
dem_file = np.load(PROCESSED / "bigsur_dem.npz")
dem = dem_file["dem_dm"] / 10.0
x0, y0, x1, y1 = dem_file["bbox_3857"]

## 3 · Analysis, every number on the map comes from here

In [ ]:
def flatten_bridges(df):
    """Elevation models read the canyon under a bridge, so we draw a straight deck across each one."""
    z = df.elev_ft.copy()
    z[df.bridge] = np.nan
    return z.interpolate(limit_direction="both")

route["z"] = flatten_bridges(route)
boston["z"] = boston.elev_ft.rolling(5, center=True, min_periods=1).mean()   # smooth out street level noise

def biggest_climb(df):
    """The largest rise from any low point to a later high point."""
    low = df.z.cummin()
    top = (df.z - low).idxmax()
    bottom = df.z[:top + 1].idxmin()
    return bottom, top, float(df.z[top] - df.z[bottom])

# Hurricane Point: from the Little Sur River to the top of the course
a = route[(route.dist_mi >= 9.5) & (route.dist_mi <= 12.5)]
hp_low, hp_top, hp_rise = biggest_climb(a)
hp_miles = route.dist_mi[hp_top] - route.dist_mi[hp_low]
# Heartbreak Hill: the last big climb on Commonwealth Avenue
b = boston[(boston.dist_mi >= 1.0) & (boston.dist_mi <= 2.0)]    # past the earlier Newton hills
h_low, h_top, h_rise = biggest_climb(b)
h_miles = boston.dist_mi[h_top] - boston.dist_mi[h_low]
ratio = hp_rise / h_rise
print(f"Route length       {route.dist_mi.max():.2f} mi  (official 26.2)")
print(f"Top of the course  {route.z.max():.0f} ft  (official 560)")
print(f"Hurricane Point    {hp_rise:.0f} ft over {hp_miles:.2f} mi, mile {route.dist_mi[hp_low]:.1f} to {route.dist_mi[hp_top]:.1f}  (official chart: 40 to 560 ft = 520)")
print(f"Heartbreak Hill    {h_rise:.0f} ft over {h_miles:.2f} mi  (published 88 to 91)")
print(f"Ratio              {ratio:.1f}x")
print(f"Total climbing     {route.z.diff().clip(lower=0).sum():.0f} ft  (published 2,182)")

## 4 · Map

In [ ]:
fig, ax = poster.page("#0b1115")
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -122.075, 36.215, -121.70, 36.565)

# terrain: dark land, ocean left empty
bad = dem > 1800
if bad.any():
    dem[bad] = ndimage.median_filter(dem, size=9)[bad]
pixel = (x1 - x0) / dem.shape[1]
shade = LightSource(azdeg=315, altdeg=38).hillshade(dem, vert_exag=2.2, dx=pixel, dy=pixel)
land = np.array([0.055, 0.08, 0.095]) + shade[..., None] * np.array([0.20, 0.25, 0.27])
is_land, _ = ndimage.label(ndimage.binary_opening(dem > 1, iterations=2))
sizes = np.bincount(is_land.ravel())
sizes[0] = 0
rgba = np.dstack([land, (is_land == sizes.argmax()).astype(float)])        # keep the mainland, drop ocean noise
ax.imshow(rgba, extent=(x0, x1, y0, y1), origin="upper", interpolation="bilinear", zorder=1)
ax.set_xlim(fx0, fx1); ax.set_ylim(fy0, fy1)

# the course, with the climb in pink
rx, ry = poster.merc(route.lon, route.lat)
ax.plot(rx, ry, color="white", lw=2.1, solid_capstyle="round", zorder=5)
climb = route.loc[hp_low:hp_top]
cx, cy = poster.merc(climb.lon, climb.lat)
poster.glow(ax, cx, cy, width=3.2, zorder=6)

# the profile, standing in the ocean beside the course. Same feet to width for both hills.
wall_x = fx0 + (fx1 - fx0) * 0.335            # where elevation is zero
feet = (fx1 - fx0) * 0.255 / 600              # page width per foot
mile = (ry.max() - ry.min()) / route.dist_mi.max()   # page height per mile
py = ry.min() + route.dist_mi.values * mile
px = wall_x - route.z.values * feet
ax.fill_betweenx(py, px, wall_x, color="white", alpha=0.11, lw=0, zorder=3)
ax.plot(px, py, color="white", lw=1.15, alpha=0.9, zorder=4)
ax.plot([wall_x, wall_x], [py.min(), py.max()], color="white", lw=0.6, alpha=0.35, zorder=4)
part = (route.index >= hp_low) & (route.index <= hp_top)
ax.fill_betweenx(py[part], px[part], wall_x, color=poster.PINK, alpha=0.5, lw=0, zorder=4)
poster.glow(ax, px[part], py[part], width=2.0, zorder=5)

# Heartbreak Hill at the same scale, sitting under the Big Sur start
bh = boston.loc[h_low:h_top]
hb_y0 = py.min() - (py.max() - py.min()) * 0.0
hy = fy0 + (fy1 - fy0) * 0.085 + (bh.dist_mi.values - bh.dist_mi.values[0]) * mile
hx = wall_x - (bh.z.values - boston.z[h_low]) * feet
ax.fill_betweenx(hy, hx, wall_x, color=poster.PINK, alpha=0.5, lw=0, zorder=4)
poster.glow(ax, hx, hy, width=2.0, zorder=5)
ax.plot([wall_x, wall_x], [hy.min(), hy.max()], color="white", lw=0.6, alpha=0.35, zorder=4)

# labels
poster.title(fig, TITLE, size=40, x=0.53, y=0.955,
             subtitle=f"The Big Sur Marathon climbs more than\nfive Heartbreaks in about two miles.", sub_size=15, sub_color="#c6d2d6")
left = fx0 + (fx1 - fx0) * 0.06
under = py[hp_low] - (fy1 - fy0) * 0.042
ax.text(left, under, "Hurricane Point", color=poster.PINK, fontproperties=poster.font(poster.SERIF, 21), va="top", zorder=8)
ax.text(left, under - (fy1 - fy0) * 0.03, f"{hp_rise:.0f} ft in {hp_miles:.1f} miles", color=poster.PAPER, fontproperties=poster.font(poster.SANS, 13), va="top", zorder=8)
low = fy0 + (fy1 - fy0) * 0.108
ax.text(left, low, "Heartbreak Hill", color=poster.PINK, fontproperties=poster.font(poster.SERIF, 16), va="top", zorder=8)
ax.text(left, low - (fy1 - fy0) * 0.024, f"{h_rise:.0f} ft in {h_miles:.1f} miles, same scale", color=poster.PAPER, fontproperties=poster.font(poster.SANS, 11), va="top", zorder=8)

def mark(mile_at, text, side="right"):
    i = (route.dist_mi - mile_at).abs().idxmin()
    mx, my = poster.merc(route.lon[i], route.lat[i])
    ax.scatter([mx], [my], s=24, color="white", zorder=9, lw=0)
    gap = (fx1 - fx0) * 0.016
    poster.halo_text(ax, mx + (gap if side == "right" else -gap), my, text, size=12, va="center", zorder=9, halo="#0e151a",
                     ha="left" if side == "right" else "right")

mark(11.0, "Taiko drummers")
mark(13.2, "A grand piano at Bixby Bridge")
mark(0, "Start, Big Sur")
mark(route.dist_mi.max(), "Finish, Carmel", side="left")
poster.credit(fig, CREDITS)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{hp_rise:.0f} ft of climbing against Boston's {h_rise:.0f}", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)